# E-commerce Business Analytics

Portfolio project untuk menganalisis performa marketplace/e-commerce: revenue, customer, product category, payment, delivery, review score, dan rekomendasi bisnis.

> Catatan: folder `data/raw` di paket ini berisi synthetic Olist-style dataset agar project bisa langsung dipelajari. Untuk versi final berbasis data publik Olist, ganti file CSV di `data/raw` dengan dataset asli dari Kaggle, lalu jalankan notebook ini dari awal.

## 1. Import library dan load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "data" / "raw"
CLEAN_DIR = BASE_DIR / "data" / "cleaned"
IMG_DIR = BASE_DIR / "images"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

orders.head()

## 2. Data quality check

Cek ukuran tabel, missing values, duplicate rows, dan tipe data.

In [ ]:
tables = {
    "orders": orders,
    "items": items,
    "customers": customers,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "translation": translation
}

for name, table in tables.items():
    print(f"\n{name}: {table.shape}")
    print("Duplicate rows:", table.duplicated().sum())
    print(table.isna().sum().sort_values(ascending=False).head(10))

## 3. Cleaning dan feature engineering

Kolom tanggal diubah ke `datetime`. Setelah itu dibuat kolom penting: `delivery_days`, `delivery_delay_days`, `is_late`, dan `order_month`.

In [ ]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["estimated_delivery_days"] = (
    orders["order_estimated_delivery_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days

orders["is_late"] = orders["delivery_delay_days"] > 0
orders["order_month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype(str)

orders[["order_id", "order_status", "delivery_days", "delivery_delay_days", "is_late", "order_month"]].head()

## 4. Join data menjadi analytical dataset

In [ ]:
df = orders.merge(customers, on="customer_id", how="left")
df = df.merge(items, on="order_id", how="left")
df = df.merge(payments, on="order_id", how="left")
df = df.merge(reviews[["order_id", "review_score", "review_comment_title", "review_comment_message"]], on="order_id", how="left")
df = df.merge(products, on="product_id", how="left")
df = df.merge(sellers, on="seller_id", how="left")
df = df.merge(translation, on="product_category_name", how="left")

df["revenue"] = df["price"].fillna(0) + df["freight_value"].fillna(0)
df["order_month_dt"] = pd.to_datetime(df["order_month"] + "-01")

df.to_csv(CLEAN_DIR / "ecommerce_final_dataset.csv", index=False)
df.shape

## 5. KPI bisnis utama

In [ ]:
delivered_items = df[df["order_status"] == "delivered"].copy()
delivered_orders = orders[orders["order_status"] == "delivered"].copy()

total_revenue = delivered_items["revenue"].sum()
total_orders = delivered_orders["order_id"].nunique()
total_customers = delivered_orders["customer_id"].nunique()
avg_order_value = total_revenue / total_orders
avg_review_score = delivered_items.drop_duplicates("order_id")["review_score"].mean()
late_delivery_rate = delivered_orders["is_late"].mean()
avg_delivery_days = delivered_orders["delivery_days"].mean()

kpi_summary = pd.DataFrame({
    "metric": [
        "Total Revenue", "Delivered Orders", "Unique Customers",
        "Average Order Value", "Average Review Score",
        "Late Delivery Rate", "Average Delivery Days"
    ],
    "value": [
        total_revenue, total_orders, total_customers,
        avg_order_value, avg_review_score,
        late_delivery_rate, avg_delivery_days
    ]
})

kpi_summary

## 6. Sales performance analysis

In [ ]:
monthly_revenue = delivered_items.groupby("order_month", as_index=False).agg(
    revenue=("revenue", "sum"),
    total_orders=("order_id", "nunique"),
    total_customers=("customer_id", "nunique"),
    avg_review_score=("review_score", "mean"),
    late_delivery_rate=("is_late", "mean")
)
monthly_revenue["average_order_value"] = monthly_revenue["revenue"] / monthly_revenue["total_orders"]
monthly_revenue.to_csv(CLEAN_DIR / "monthly_revenue.csv", index=False)

monthly_revenue.head()

In [ ]:
plt.figure(figsize=(11,5))
plt.plot(monthly_revenue["order_month"], monthly_revenue["revenue"] / 1_000_000, marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Order Month")
plt.ylabel("Revenue (Millions)")
plt.xticks(rotation=45, ha="right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(IMG_DIR / "monthly_revenue_trend.png", bbox_inches="tight")
plt.show()

## 7. Product category analysis

In [ ]:
category_analysis = delivered_items.groupby("product_category_name_english", as_index=False).agg(
    revenue=("revenue", "sum"),
    total_orders=("order_id", "nunique"),
    total_items=("order_item_id", "count"),
    avg_price=("price", "mean"),
    avg_freight=("freight_value", "mean"),
    avg_review_score=("review_score", "mean"),
    late_delivery_rate=("is_late", "mean")
).sort_values("revenue", ascending=False)

category_analysis.to_csv(CLEAN_DIR / "category_analysis.csv", index=False)
category_analysis.head(10)

In [ ]:
top_categories = category_analysis.head(10).sort_values("revenue")

plt.figure(figsize=(10,6))
plt.barh(top_categories["product_category_name_english"], top_categories["revenue"] / 1_000_000)
plt.title("Top 10 Categories by Revenue")
plt.xlabel("Revenue (Millions)")
plt.ylabel("Product Category")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(IMG_DIR / "top_categories_by_revenue.png", bbox_inches="tight")
plt.show()

## 8. Customer and location analysis

In [ ]:
state_analysis = delivered_items.groupby("customer_state", as_index=False).agg(
    revenue=("revenue", "sum"),
    total_orders=("order_id", "nunique"),
    total_customers=("customer_id", "nunique"),
    avg_delivery_days=("delivery_days", "mean"),
    late_delivery_rate=("is_late", "mean"),
    avg_review_score=("review_score", "mean")
).sort_values("revenue", ascending=False)
state_analysis["average_order_value"] = state_analysis["revenue"] / state_analysis["total_orders"]

state_analysis.to_csv(CLEAN_DIR / "state_analysis.csv", index=False)
state_analysis.head(10)

## 9. Delivery performance vs review score

In [ ]:
review_delivery = delivered_items.drop_duplicates("order_id").groupby("review_score", as_index=False).agg(
    total_orders=("order_id", "nunique"),
    avg_delivery_days=("delivery_days", "mean"),
    avg_delay_days=("delivery_delay_days", "mean"),
    late_delivery_rate=("is_late", "mean")
)

review_delivery.to_csv(CLEAN_DIR / "review_delivery_analysis.csv", index=False)
review_delivery

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(review_delivery["review_score"].astype(str), review_delivery["late_delivery_rate"] * 100)
plt.title("Late Delivery Rate by Review Score")
plt.xlabel("Review Score")
plt.ylabel("Late Delivery Rate (%)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(IMG_DIR / "late_delivery_by_review_score.png", bbox_inches="tight")
plt.show()

## 10. Business insights

Gunakan pola ini untuk menulis insight di README, executive summary, dan presentasi interview:

1. **Revenue concentration**: revenue didominasi beberapa kategori, sehingga kategori prioritas perlu dipantau dari sisi stok, seller quality, dan fulfilment.
2. **Delivery quality matters**: late delivery rate jauh lebih tinggi pada review score rendah, sehingga pengiriman adalah faktor penting untuk customer satisfaction.
3. **Regional focus**: state dengan revenue terbesar bisa dijadikan target marketing dan optimasi logistik.
4. **Seller monitoring**: seller dengan revenue tinggi tetapi review rendah/late rate tinggi perlu masuk daftar evaluasi SLA.

## 11. Export final analytical dataset

In [ ]:
orders.to_csv(CLEAN_DIR / "orders_cleaned.csv", index=False)
df.to_csv(CLEAN_DIR / "ecommerce_final_dataset.csv", index=False)
print("Done. Cleaned files saved to:", CLEAN_DIR)